# Construcción de una Biblioteca Reutilizable de Tarificación Actuarial con PROC FCMP

## Resumen Ejecutivo

Una aseguradora de propiedad y accidentes encapsula sus fórmulas de tarificación centrales — prima pura, carga de gastos/riesgo, combinación de credibilidad de fluctuación limitada y estimación de reserva descontada — como funciones personalizadas y una subrutina de múltiples salidas en **PROC FCMP**. La biblioteca compilada se registra mediante la opción de sistema `CMPLIB=` y luego se llama fila por fila desde un paso DATA que tarifica una cartera sintética de 100 pólizas. Cada cifra tarificada en este cuaderno — el factor de credibilidad `z`, la prima pura combinada, la prima cargada y la reserva de siniestro a valor presente — es producida por estas rutinas compiladas, no por aritmética en línea. El resultado: la razón de siniestralidad implícita se ubica en 60.5% (rural), 55.8% (suburbano) y 63.6% (urbano) — cómodamente por debajo del 100% en cada segmento, confirmando que la prima cargada cubre la pérdida esperada mientras el paso de tarificación permanece limpio y auditable.

## Fuentes de Datos

| Conjunto de Datos | Filas | Descripción | Variables Clave |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Cartera sintética de pólizas de P&C vigentes generada en línea con `rand()` | `policy_id`, `region` (urbano/suburbano/rural), `years_insured`, `exposure` (años-vehículo), `claim_count` (Poisson), `incurred_loss` (severidad gamma x cantidad), `class_pure_prem` (tarifa manual por región) |

El paso DATA recorre un rango más amplio de `policy_id`, pero este entorno se ejecuta sin licencia, por lo que la salida se limita a las primeras **100 observaciones** — la cartera tarificada abajo refleja esas 100 pólizas.

# Construcción de una Biblioteca Reutilizable de Tarificación Actuarial con PROC FCMP

Los actuarios repiten los mismos cálculos en trabajos de tarificación, reservas e informes: convertir pérdidas en una *prima pura*, aplicar *cargas de gastos y riesgo* para llegar a una prima cargada, combinar la experiencia propia de una póliza con una tarifa de clase mediante *credibilidad*, y *descontar* flujos de caja futuros a valor presente. Volver a escribir estas fórmulas en cada paso DATA invita a errores de copiar y pegar y hace dolorosos los cambios de supuestos.

**PROC FCMP** (el compilador de funciones de SAS) nos permite definir cada fórmula una sola vez como una función o subrutina con nombre, almacenar las rutinas compiladas en una biblioteca y luego llamarlas como cualquier función incorporada de SAS. En este cuaderno:

1. Compilamos una pequeña biblioteca de funciones actuariales con `PROC FCMP`.
2. La registramos para la sesión con la opción de sistema `CMPLIB=`.
3. Generamos una cartera sintética de propiedad y accidentes.
4. Tarificamos cada póliza llamando a nuestras funciones personalizadas y una subrutina de múltiples salidas desde un único paso DATA.
5. Resumimos e interpretamos la cartera tarificada.

## Paso 1 — Generar una cartera sintética de pólizas

Simulamos un libro de pólizas de auto vigentes (las primeras 100 se tarifican abajo bajo el límite del modo sin licencia). Cada póliza pertenece a una región de tarifa con su propia *prima pura* manual (pérdida esperada por año-vehículo). Las cantidades de siniestros siguen un proceso de Poisson escalado por la exposición, y las severidades siguen una distribución gamma, por lo que `incurred_loss` es una pérdida compuesta realista (Poisson x gamma). `years_insured` impulsará más adelante el peso de credibilidad. Una semilla fija mediante `call streaminit` hace que los datos sean reproducibles.

In [1]:
DATOS portfolio;
    LLAMAR streaminit(20260531);
    LONGITUD region $9;
    HACER policy_id = 10001 HASTA 12000;
        /* Asignar una región de tarifa */
        u = rand('uniform');
        SI u < 0.40 ENTONCES HACER; region = 'urbano';    base_pp = 820; lambda = 0.18; END;
        SINO SI u < 0.75 ENTONCES HACER; region = 'suburbano'; base_pp = 540; lambda = 0.11; END;
        SINO HACER; region = 'rural';    base_pp = 360; lambda = 0.07; END;

        /* Antigüedad (años asegurado) y exposición (años-vehículo) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Proceso compuesto de siniestros: frecuencia Poisson, severidad gamma */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        HACER c = 1 HASTA claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        END;
        incurred_loss = round(incurred_loss, 0.01);

        /* Prima pura de clase manual para la exposición de esta póliza */
        class_pure_prem = round(base_pp * exposure, 0.01);

        SALIDA;
    END;
    MANTENER policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=portfolio(obs=8) noobs ETIQUETA;
    ETIQUETA policy_id='ID de Póliza'
          region='Región'
          years_insured='Años Asegurado'
          exposure='Exposición (años-vehículo)'
          claim_count='Cantidad de Siniestros'
          incurred_loss='Pérdida Incurrida'
          class_pure_prem='Prima Pura de Clase';
    TÍTULO 'Primeras 8 Pólizas Simuladas';
EJECUTAR;

                                              Primeras 8 Pólizas Simuladas                                              

 ID de Póliza     Región   Años Asegurado     Exposición (años-vehículo)  Cantidad de Siniestros   Pérdida Incurrida  Prima Pura de Clase
        10001  rural                    5                           1.36                       0                   0                489.6
        10002  urbano                   8                           0.97                       1             3432.28                795.4
        10003  urbano                   2                           1.53                       2             7155.92               1254.6
        10004  suburbano                9                            2.4                       0                   0                 1296
        10005  rural                    5                           0.99                       0                   0                356.4
        10006  suburbano                1         


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.43 seconds
  cpu   0.43 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Paso 2 — Compilar la biblioteca de funciones actuariales

Ahora el corazón del cuaderno. `PROC FCMP` con `OUTLIB=work.actfuncs.pricing` compila cuatro funciones y una subrutina en el paquete `pricing` del conjunto de datos `work.actfuncs`:

- **`pure_premium`** — pérdida observada por unidad de exposición (frecuencia x severidad combinadas).
- **`credibility_z`** — factor de credibilidad de fluctuación limitada `Z = sqrt(n / (n + k))`, donde `n` son los años-exposición de la póliza y `k` es una constante de ajuste.
- **`charged_premium`** — aplica una carga de riesgo proporcional y una razón de gastos fija a un costo de pérdida para llegar a la prima que realmente cobramos.
- **`pv_reserve`** — valor presente de un pago de siniestro futuro, `FV / (1+r)**t`, usado para descontar las reservas de caso.
- **`blend_premium`** (una SUBROUTINE) — usa `OUTARGS` para devolver *dos* valores a la vez: la prima pura ponderada por credibilidad y el factor de credibilidad que usó, de modo que el paso DATA obtiene ambos en una sola llamada.

Cada rutina se cierra con `ENDSUB`, y la subrutina nombra sus argumentos escribibles con `OUTARGS`.

In [2]:
PROCEDIMIENTO fcmp outlib=work.actfuncs.pricing;

    /* Prima pura observada: costo de pérdida por unidad de exposición */
    function pure_premium(loss, exposure);
        SI exposure <= 0 ENTONCES RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Credibilidad de fluctuación limitada: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        SI n_years <= 0 ENTONCES RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Prima cargada = costo de pérdida ajustado por carga de riesgo + gastos */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        SI expense_ratio >= 1 ENTONCES RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Valor presente de un pago de siniestro futuro descontado t años a tasa r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Combinación de credibilidad: devuelve la prima pura ponderada Y la Z usada */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

EJECUTAR;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Paso 3 — Registrar la biblioteca con CMPLIB=

No basta con compilar las rutinas; hay que indicarle a SAS dónde encontrarlas cuando un paso DATA o procedimiento posterior haga referencia a un nombre que no reconoce como incorporado. La opción `CMPLIB=` apunta al conjunto de datos (no al nombre de paquete de tres niveles) que contiene el código compilado. Después de esta sentencia `OPTIONS`, cada función y subrutina anterior es invocable por su nombre.

In [3]:
OPCIONES cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Paso 4 — Tarificar la cartera con las funciones personalizadas

El paso DATA de tarificación ahora está casi libre de aritmética — la intención actuarial se lee directamente en los nombres de las funciones. Para cada póliza:

1. Calculamos la prima pura observada propia de la póliza con `pure_premium`.
2. Llamamos a la subrutina `blend_premium` para ponderarla por credibilidad contra la tarifa de clase regional, recuperando tanto el costo de pérdida combinado como el factor de credibilidad `z` mediante `OUTARGS`.
3. Ajustamos el costo de pérdida combinado a una prima cargada con una carga de riesgo del 12% y una razón de gastos del 25% mediante `charged_premium`.
4. Estimamos la reserva de caso aún abierta como el 35% de la pérdida incurrida de la póliza y la descontamos tres años al 4% a valor presente con `pv_reserve`.

Nótese cómo los argumentos de salida de la subrutina (`blended_pp`, `z`) deben inicializarse antes del `CALL`. La reserva a valor presente varía de póliza a póliza porque depende de la pérdida incurrida real de cada póliza — las pólizas sin siniestros llevan una reserva de cero, por lo que su `reserve_pv` también es cero.

In [4]:
DATOS rated;
    ESTABLECER portfolio;

    /* 1. Experiencia de pérdida propia de la póliza como prima pura */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Ponderar por credibilidad la experiencia propia contra la tarifa
          de clase. k = 4 años-exposición para una credibilidad casi plena. */
    blended_pp = .;
    z = .;
    LLAMAR blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Convertir el costo de pérdida combinado (por año-vehículo) en
          prima cargada */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. La reserva de caso pendiente = 35% de la pérdida incurrida, con
          liquidación esperada en 3 años; descontarla a valor presente al 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    MANTENER policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=rated(obs=10) noobs ETIQUETA;
    TÍTULO 'Cartera Tarifada (primeras 10 pólizas)';
    VAR policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
    ETIQUETA policy_id='ID de Póliza'
          region='Región'
          years_insured='Años Asegurado'
          exposure='Exposición (años-vehículo)'
          own_pp='Prima Pura Propia'
          blended_pp='Prima Pura Combinada'
          z='Factor de Credibilidad (Z)'
          premium='Prima Cargada'
          reserve_pv='VP de Reserva';
EJECUTAR;

                                         Cartera Tarifada (primeras 10 pólizas)                                         

 ID de Póliza     Región   Años Asegurado     Exposición (años-vehículo)  Prima Pura Propia  Prima Pura Combinada  Factor de Credibilidad (Z)  Prima Cargada  VP de Reserva
        10001  rural                    5                           1.36                  0                 91.67                       0.745         186.18              0
        10002  urbano                   8                           0.97            3538.43               3039.59                       0.816        4402.95        1067.95
        10003  urbano                   2                           1.53            4677.07               3046.88                       0.577        6961.51        2226.55
        10004  suburbano                9                            2.4                  0                 90.69                       0.832         325.03              0
        10005  rur


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Paso 5 — Resumir la cartera tarificada

Con cada póliza tarificada mediante la misma biblioteca compilada, podemos agregar la cartera por región. Reportamos la prima cargada media, el factor de credibilidad medio, la pérdida incurrida total y la reserva de caso total a valor presente, y luego derivamos la razón de siniestralidad implícita por segmento para ver si la prima cargada cubre la pérdida esperada.

In [5]:
PROCEDIMIENTO MEDIAS DATOS=rated mean sum maxdec=2 nonobs;
    CLASE region;
    VAR premium z incurred_loss reserve_pv;
    ETIQUETA region='Región'
          premium='Prima Cargada'
          z='Factor de Credibilidad (Z)'
          incurred_loss='Pérdida Incurrida'
          reserve_pv='VP de Reserva';
    TÍTULO 'Resumen de la Cartera por Región de Tarifa';
EJECUTAR;

/* La razón de siniestralidad implícita = pérdida incurrida / prima cargada,
   más la reserva a valor presente que lleva el segmento, por región. */
PROCEDIMIENTO SQL;
    TÍTULO 'Razón de Siniestralidad Implícita y VP de Reserva por Región';
    SELECCIONAR region label='Región',
           count(*)                              AS n_policies label='Núm. Pólizas',
           sum(incurred_loss)                    AS total_loss     label='Pérdida Incurrida Total' FORMATO=dollar12.2,
           sum(premium)                          AS total_premium  label='Prima Total'             FORMATO=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     label='Razón de Siniestralidad'  FORMATO=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve label='Reserva VP Total'       FORMATO=dollar12.2
    DESDE rated
    GROUP POR region
    ORDER POR region;
QUIT;
TÍTULO;

                                       Resumen de la Cartera por Región de Tarifa                                       

                                                  The MEANS Procedure

                                       Analysis Variable : premium Prima Cargada

        Región              Mean            Sum
        ---------------------------------------
        rural             476.61       11438.62
        suburbano         813.04       34147.74
        urbano           1987.17       67563.93
        ---------------------------------------

                                    Analysis Variable : z Factor de Credibilidad (Z)

        Región              Mean            Sum
        ---------------------------------------
        rural               0.71          17.14
        suburbano           0.68          28.36
        urbano              0.70          23.90
        ---------------------------------------

                                  Analysis Variable : incurre


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretación de los resultados

El paso DATA de tarificación nunca escribe una sola fórmula de descuento o credibilidad en línea — solo llama a `pure_premium`, `blend_premium`, `charged_premium` y `pv_reserve`. Ese es el beneficio de **PROC FCMP**: los supuestos actuariales viven en una sola biblioteca compilada que puede probarse unitariamente, versionarse y reutilizarse en trabajos de tarificación, reservas e informes. Cambiar la constante de credibilidad `k`, la carga de riesgo o la razón de gastos es una sola edición en la biblioteca, no una búsqueda en cada programa.

Leyendo la cartera tarificada y la agregación regional:

- **La credibilidad (`z`)** aumenta con `years_insured`, exactamente como dicta la fórmula de fluctuación limitada `Z = sqrt(n / (n + k))`. Entre las primeras diez pólizas, la póliza suburbana de un año (10006) obtiene `z = 0.447`, la póliza urbana de dos años (10003) `z = 0.577`, mientras que la póliza suburbana de nueve años (10004) alcanza `z = 0.832`. Las pólizas con poca experiencia se acercan a la tarifa de clase regional; las de mayor antigüedad se apoyan en sus propias pérdidas.
- **La combinación en acción:** las pólizas sin siniestros (la mayoría del libro) tienen `own_pp = 0`, por lo que `blend_premium` devuelve una `blended_pp` cercana a `(1 - z)` veces la tarifa de clase — por ejemplo, la póliza 10001 (rural, `z = 0.745`) queda en `91.67` frente a una tarifa de clase de `360`/año-vehículo. Las dos pólizas urbanas que sí tuvieron siniestros, 10002 y 10003, en cambio elevan su costo de pérdida combinado hacia su propia experiencia alta (`3039.59` y `3046.88`).
- **La prima cargada** queda por encima del costo de pérdida combinado porque `charged_premium` agrega una carga de riesgo del 12% y ajusta por una razón de gastos del 25%, un multiplicador fijo `1.12 / 0.75 = 1.493` sobre el costo de pérdida. La prima cargada media es `476.61` (rural), `813.04` (suburbano) y `1,987.17` (urbano).
- **Las reservas descontadas:** `pv_reserve` descuenta la reserva de caso pendiente de cada póliza (35% de la pérdida incurrida) tres años al 4%, un factor de valor presente de `0.889`. Las pólizas sin siniestros llevan `reserve_pv = 0`; los dos reclamantes urbanos aportan `1,067.95` y `2,226.55`. Agregado, el libro mantiene una reserva a valor presente de `$2,151.56` (rural), `$5,932.52` (suburbano) y `$13,364.13` (urbano).
- **Las razones de siniestralidad implícitas** se ubican en 60.5% (rural), 55.8% (suburbano) y 63.6% (urbano) — todas cómodamente por debajo del 100%, por lo que la prima cargada cubre la pérdida esperada en cada segmento. El segmento urbano corre el más caliente, impulsado por sus dos grandes pérdidas simuladas; una revisión de tarifa real pondría a prueba si esa señal persiste en más años de accidentes antes de ajustar la tarifa manual.

La subrutina `blend_premium` también demuestra el idioma `OUTARGS` para devolver múltiples resultados desde una sola `CALL` — el costo de pérdida combinado y el factor de credibilidad `z` regresan juntos — evitando llamadas de función separadas y manteniendo compacta y auditable la lógica de tarificación por póliza.